<a href="https://colab.research.google.com/github/rawanahmedashraf-blip/hidden-gaps-students-opportunities/blob/main/notebooks/%20job%20market%20data%20phase0_code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
import pandas as pd
import numpy as np
import re

# 1. تحميل الملف
df = pd.read_excel('/content/The Hidden Gap_ Effort vs Opportunities  (Responses) (1).xlsx')
# 2. منطق إعادة تسمية الأعمدة (Substring Matching) لضمان تجاوز مشكلات المسافات وعلامات الترقيم
rename_logic = {
    'ساعة أسبوعيًا': 'learning_hours',
    'قيّم مستوى استعدادك': 'interview_readiness',
    'شخص يعمل في مجالك': 'mentor_contact',
    'عدد الفرص': 'apps_count',
    'تم قبولك': 'acceptances_count',
    'في حالة الرفض': 'feedback_frequency',
    'متطلباتها واضحة': 'requirements_clarity',
    'فاهم متطلبات سوق العمل': 'market_understanding',
    'عدم قبولك': 'perceived_barriers',
    'دفعت مقابل كورس': 'paid_for_training',
    'فعاليات مهنية': 'events_attended',
    'إيه الناقصك': 'can_identify_gaps',
    'إحباط في التقديم': 'frustration_sources',
    'تقرير شخصي يطابق': 'wants_skill_report',
    'أكبر تحدٍ يمنعك': 'biggest_challenge',
    '[Python]': 'python_lvl',
    '[SQL]': 'sql_lvl',
    '[Excel]': 'excel_lvl',
    'Statistics': 'stats_lvl',
    'Machine Learning': 'ml_knowledge',
    'عدد المشاريع العملية': 'projects_count',
    'مسابقات تحليل بيانات': 'kaggle_participation',
    'بتعرف عن الفرص': 'info_sources',
    'نوع الجامعة': 'university_type',
    'موقع دراستك الحالي': 'location',
    'مرحلتك الدراسية': 'academic_year',
    'الكلية': 'faculty',
    'التخصص/القسم': 'major',
    'حساب GitHub': 'has_portfolio',
    'عرض البيانات': 'viz_tools'
}

for col in df.columns:
    for key, new_name in rename_logic.items():
        if key in col:
            df.rename(columns={col: new_name}, inplace=True)

# 3. تحويل مستويات المهارات إلى مقياس رقمي (0-4) ضروري لحساب الـ Gap Score لاحقاً
skill_map = {'لا أعرفها': 0, 'مبتدئ': 1, 'مستوى متوسط': 2, 'متقدم': 3, 'خبير': 4}
skill_cols = ['python_lvl', 'sql_lvl', 'excel_lvl', 'stats_lvl']

for col in skill_cols:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip().map(skill_map).fillna(0).astype(int)

print("تمت إعادة التسمية وتعديل المهارات بنجاح. الأعمدة الحالية:")
print(df.columns.tolist())
df['has_python'] = df['python_lvl'] >= 2
df['has_sql'] = df['sql_lvl'] >= 2
df['has_excel'] = df['excel_lvl'] >= 2
df['has_statistics'] = df['stats_lvl'] >= 2

# 2. معالجة أدوات عرض البيانات (Tableau / Power BI)
if 'viz_tools' in df.columns:
    df['has_tableau'] = df['viz_tools'].str.contains('Tableau', case=False, na=False)
    df['has_powerbi'] = df['viz_tools'].str.contains('Power BI', case=False, na=False)
    # يعتبر الطالب ممتلكاً لأدوات العرض إذا اختار Tableau أو Power BI
    df['has_any_viz_tool'] = df['viz_tools'].str.contains('Tableau|Power BI', case=False, na=False)
    # 3. معالجة الرينجات (المجهود، التقديم، القبول، الفعاليات) باستخدام المنطق الإحصائي
hours_map = {'أقل من 2': 1, '2–5': 3, '5–10': 7, 'أكثر من 10': 12}
apps_map = {'0': 0, '1-3': 2, '4-10': 7, '11-25': 18, '+25': 30}
accept_map = {'ولا وحده': 0, 'ولا واحدة': 0, '1': 1, '2 -3': 2, '2-3': 2, 'أكتر': 4}
events_map = {'لا': 0, 'لم تتوفر فرص في منطقتي': 0, 'مرة أو مرتين': 1, 'أكثر من 3 مرات': 3}

# تطبيق التحويل مع الاحتفاظ بالأعمدة الأصلية وإنشاء أعمدة رقمية جديدة (_num)
if 'learning_hours' in df.columns:
    df['learning_hours_num'] = df['learning_hours'].str.strip().map(hours_map).fillna(0).astype(int)

if 'apps_count' in df.columns:
    df['apps_count_num'] = df['apps_count'].str.strip().map(apps_map).fillna(0).astype(int)

if 'acceptances_count' in df.columns:
    df['accept_count_num'] = df['acceptances_count'].str.strip().map(accept_map).fillna(0).astype(int)

if 'events_attended' in df.columns:
    df['events_num'] = df['events_attended'].str.strip().map(events_map).fillna(0).astype(int)

# 4. التصنيفات الفئوية (الموقع، الكلية، التخصص)
# توحيد المواقع: القاهرة/الإسكندرية مقابل الأقاليم
cairo_alex = ['القاهرة', 'الجيزة', 'الاسكندرية', 'الإسكندرية', 'cairo', 'giza', 'alex']
if 'location' in df.columns:
    df['region_category'] = df['location'].apply(lambda x: 'Cairo/Alex' if any(c in str(x) for c in cairo_alex) else 'Provinces')
    df['is_cairo'] = df['region_category'] == 'Cairo/Alex' # Binary flag لتسهيل التحليل لاحقاً

# تصنيف الكليات لتجميعها في قطاعات قابلة للمقارنة
faculty_map = {
    'حاسبات ومعلومات / علوم الحاسب': 'Tech/CS',
    'ذكاء اصطناعي / علوم البيانات': 'Tech/CS',
    'تكنولوجيا المعلومات (IT)': 'Tech/CS',
    'تجارة / إدارة أعمال': 'Business',
    'BIS': 'Business/Tech',
    'هندسة': 'Engineering',
    'علوم': 'Science',
    'اقتصاد وعلوم سياسية': 'Business'
}
if 'faculty' in df.columns:
    df['faculty_category'] = df['faculty'].str.strip().map(faculty_map).fillna('Other')
    # تحديد التخصصات ذات الصلة المباشرة بالمجال (1 = صلة مباشرة، 0 = صلة غير مباشرة/أخرى)
def check_relevance(major):
    major = str(major).lower()
    related_keywords = ['cs', 'computer', 'ai', 'data', 'ذكاء', 'بيانات', 'it', 'is', 'bis', 'info', 'stat', 'إحصاء', 'software', 'نظم', 'حاسب']
    return 1 if any(k in major for k in related_keywords) else 0

if 'major' in df.columns:
    df['is_related_major'] = df['major'].apply(check_relevance)

print("تم تنفيذ المرحلة 1.1 بالكامل. البيانات النظيفة جاهزة للتحليل الوصفي.")
print("\nعينة من البيانات النظيفة:")
print(df[['has_python', 'has_tableau', 'apps_count_num', 'region_category', 'is_related_major']].head())

df.head(5)
import pandas as pd
import numpy as np

# هذا الكود يكتب بعد الانتهاء من عملية تغيير الأسماء (Rename Logic) والتحويلات الأساسية

print("=" * 80)
print("STUDENT DATA - COMPLETE FEATURE ENGINEERING (Claude V2.0)")
print("=" * 80)

# 1. Machine Learning
if 'ml_knowledge' in df.columns:
    df['has_ml'] = df['ml_knowledge'].astype(str).str.contains('نعم', na=False)

# 2. Projects Count (معالجة ذكية للقيم الموجودة في الصورة لتفادي أخطاء المسافات)
def parse_projects(val):
    val = str(val)
    if 'لم أنجز' in val: return 0
    if 'واحد' in val: return 1
    if '2' in val and '3' in val: return 2.5  # نقطة المنتصف
    if '4' in val and '5' in val: return 4.5  # نقطة المنتصف
    if 'أكثر' in val or '5' in val: return 7   # تقدير لأكثر من 5
    return 0

if 'projects_count' in df.columns:
    df['projects_num'] = df['projects_count'].apply(parse_projects)
    df['has_projects'] = df['projects_num'] >= 1

# 3. Kaggle Participation
if 'kaggle_participation' in df.columns:
    df['has_kaggle'] = df['kaggle_participation'].astype(str).str.contains('نعم', na=False)

# 4. Portfolio
if 'has_portfolio' in df.columns:
    df['has_portfolio_flag'] = df['has_portfolio'].astype(str).str.contains('نعم', na=False)

# 5. Mentor Contact
if 'mentor_contact' in df.columns:
    df['has_mentor'] = df['mentor_contact'].astype(str).str.contains('نعم', na=False)
    df['tried_mentor'] = df['mentor_contact'].astype(str).str.contains('حاولت', na=False)

# 6. Paid Training (تم الضبط على القيم الحقيقية بدقة)
if 'paid_for_training' in df.columns:
    df['paid_training_flag'] = df['paid_for_training'].astype(str).str.contains('نعم', na=False)
    df['cant_afford'] = df['paid_for_training'].astype(str).str.contains('لا استطيع تحمل التكلفة', na=False)
    df['no_value'] = df['paid_for_training'].astype(str).str.contains('لا اري قيمة لذلك', na=False)

# 7. Interview Readiness
if 'interview_readiness' in df.columns:
    df['high_interview_readiness'] = pd.to_numeric(df['interview_readiness'], errors='coerce').fillna(0) >= 4

# 8. Academic Background (تمت إضافة الفئات المخفية)
if 'university_type' in df.columns:
    df['is_government_univ'] = df['university_type'].astype(str).str.contains('حكومية', na=False)
    df['is_private_univ'] = df['university_type'].astype(str).str.contains('خاصة', na=False)
    # إضافة الفئات الموجودة في الداتا وتجاهلتها خطة Claude
    df['is_national_univ'] = df['university_type'].astype(str).str.contains('اهلية/دولية', na=False)
    df['is_diploma'] = df['university_type'].astype(str).str.contains('دبلومة', na=False)

if 'academic_year' in df.columns:
    # تم التحديد بناءً على القيم الحقيقية (خريج حديث، رابعة، خامسة)
    df['is_graduate'] = df['academic_year'].astype(str).str.contains('خريج حديث', na=False)
    df['is_senior'] = df['academic_year'].astype(str).str.contains('رابعة|خامسة', na=False)

# ---------------------------------------------------------
# 9. Complete Skill Supply Calculation (المرحلة الأولى)
# ---------------------------------------------------------
skill_supply = {
    # Technical Skills
    'Python': (df['has_python'].sum() / len(df)) * 100,
    'SQL': (df['has_sql'].sum() / len(df)) * 100,
    'Excel': (df['has_excel'].sum() / len(df)) * 100,
    'Statistics': (df['has_statistics'].sum() / len(df)) * 100,
    'Tableau': (df['has_tableau'].sum() / len(df)) * 100,
    'Power BI': (df['has_powerbi'].sum() / len(df)) * 100,
    'Visualization (General)': (df['has_any_viz_tool'].sum() / len(df)) * 100,
    'Machine Learning': (df['has_ml'].sum() / len(df)) * 100,

    # Portfolio & Experience Indicators
    'Portfolio': (df['has_portfolio_flag'].sum() / len(df)) * 100,
    'Projects (1+)': (df['has_projects'].sum() / len(df)) * 100,
    'Kaggle': (df['has_kaggle'].sum() / len(df)) * 100,
}

print("\nSTUDENT SKILL SUPPLY (%):")
for skill, pct in skill_supply.items():
    print(f"  {skill:30s}: {pct:5.1f}%")

# حفظ بيانات العرض للاستخدام في تحليل الفجوة
supply_df = pd.DataFrame(skill_supply.items(), columns=['Skill', 'Student Supply %'])
supply_df.to_csv('student_skill_supply_COMPLETE.csv', index=False)

# حفظ الداتا سيت النهائية والشاملة
df.to_csv('student_data_FINAL.csv', index=False)

print("\n✅ تم حفظ الملفات:")
print("1. student_skill_supply_COMPLETE.csv (لاستخدامه في تحليل الفجوة)")
print("2. student_data_FINAL.csv (البيانات الشاملة للمراحل القادمة)")



تمت إعادة التسمية وتعديل المهارات بنجاح. الأعمدة الحالية:
['Timestamp', 'university_type', 'location', 'academic_year', 'faculty', 'major', 'learning_hours', 'نوع التطوير اللي بتركز عليه أكتر', 'interview_readiness', 'mentor_contact', 'apps_count', 'acceptances_count', 'feedback_frequency', 'requirements_clarity', 'market_understanding', 'perceived_barriers', 'paid_for_training', 'events_attended', 'can_identify_gaps', 'frustration_sources', 'wants_skill_report', 'biggest_challenge', 'python_lvl', 'sql_lvl', 'excel_lvl', 'stats_lvl', 'viz_tools', 'ml_knowledge', 'projects_count', 'kaggle_participation', 'has_portfolio', 'info_sources']
تم تنفيذ المرحلة 1.1 بالكامل. البيانات النظيفة جاهزة للتحليل الوصفي.

عينة من البيانات النظيفة:
   has_python  has_tableau  apps_count_num region_category  is_related_major
0       False        False               0       Provinces                 1
1       False        False               0       Provinces                 0
2       False        False   